In [1]:
from __future__ import annotations

import json
import textwrap
import time
import zipfile
from datetime import datetime, timezone
from io import BytesIO
from uuid import uuid4

import pandas as pd
from google.api_core.exceptions import Conflict, NotFound
from google.cloud import bigquery
from google.cloud import storage

In [2]:
PROJECT_ID = "leafy-guide-497515-m4"
LOCATION = "us-central1"

BUCKET_NAME = "leafy-guide-497515-m4-vector-assets"

SOURCE_RAG_DATASET_ID = "weaviate_mentor_rag_backend"
SOURCE_CHUNK_TABLE_ID = "weaviate_mentor_chunks"

VOICE_DATASET_ID = "weaviate_voice_widget_cloud_run"

VOICE_ASSET_TABLE_ID = "voice_widget_assets"
VOICE_TEST_TABLE_ID = "voice_widget_test_requests"

SERVICE_NAME = "weaviate-voice-mentor-api"
ARTIFACT_REPOSITORY = "voice-mentor"

TEXT_MODEL = "gemini-2.5-flash"
TEXT_EMBEDDING_MODEL = "text-embedding-005"

TOP_K_DEFAULT = 8

ALLOWED_ORIGIN = "https://weaviate-voice-mentor.lovable.app"

DEFAULT_LANGUAGE_CODE = "pl-PL"
DEFAULT_ALTERNATIVE_LANGUAGE_CODE = "en-US"

GCS_BACKEND_PREFIX = "weaviate-voice-widget-cloud-run/backend"
GCS_WIDGET_PREFIX = "weaviate-voice-widget-cloud-run/widget"
GCS_DEPLOY_PREFIX = "weaviate-voice-widget-cloud-run/deploy"
GCS_SUMMARY_PREFIX = "weaviate-voice-widget-cloud-run/summaries"

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Location:", LOCATION)
print("Bucket:", BUCKET_NAME)
print("Source RAG dataset:", SOURCE_RAG_DATASET_ID)
print("Source chunk table:", SOURCE_CHUNK_TABLE_ID)
print("Voice dataset:", VOICE_DATASET_ID)
print("Service name:", SERVICE_NAME)
print("Allowed origin:", ALLOWED_ORIGIN)

Configuration loaded.
Project: leafy-guide-497515-m4
Location: us-central1
Bucket: leafy-guide-497515-m4-vector-assets
Source RAG dataset: weaviate_mentor_rag_backend
Source chunk table: weaviate_mentor_chunks
Voice dataset: weaviate_voice_widget_cloud_run
Service name: weaviate-voice-mentor-api
Allowed origin: https://weaviate-voice-mentor.lovable.app


In [3]:
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)
bucket.reload()

bigquery_client = bigquery.Client(project=PROJECT_ID)

BUCKET_LOCATION = bucket.location

if BUCKET_LOCATION in {"US", "EU"}:
    BIGQUERY_LOCATION = BUCKET_LOCATION
else:
    BIGQUERY_LOCATION = BUCKET_LOCATION.lower()

source_chunk_table_ref = (
    f"{PROJECT_ID}."
    f"{SOURCE_RAG_DATASET_ID}."
    f"{SOURCE_CHUNK_TABLE_ID}"
)

print("Clients created.")
print("Bucket exists:", bucket.exists())
print("Bucket location:", BUCKET_LOCATION)
print("BigQuery location:", BIGQUERY_LOCATION)
print("Source chunk table:", source_chunk_table_ref)

Clients created.
Bucket exists: True
Bucket location: EU
BigQuery location: EU
Source chunk table: leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_chunks


In [4]:
source_table = bigquery_client.get_table(source_chunk_table_ref)

print("Source table exists.")
print("Rows:", source_table.num_rows)
print("Location:", source_table.location)

sql = f"""
SELECT
  COUNT(*) AS chunk_count,
  MIN(global_chunk_number) AS first_chunk,
  MAX(global_chunk_number) AS last_chunk,
  ARRAY_AGG(DISTINCT source_title ORDER BY source_title LIMIT 20) AS sections
FROM `{source_chunk_table_ref}`
"""

source_summary = list(
    bigquery_client.query(
        sql,
        location=source_table.location,
    )
)[0]

print("Chunk count:", source_summary.chunk_count)
print("First chunk:", source_summary.first_chunk)
print("Last chunk:", source_summary.last_chunk)
print("Sections:")
for section in source_summary.sections:
    print("-", section)

Source table exists.
Rows: 19
Location: EU
Chunk count: 19
First chunk: 1
Last chunk: 19
Sections:
- 1. What Weaviate is
- 10. Metadata filters
- 11. References
- 12. Weaviate Cloud vs local Docker
- 13. Python client pattern
- 14. Common beginner mistakes
- 15. JSON, dict, and serialization
- 16. Weaviate and SQL mental model
- 17. When to use Weaviate
- 18. Voice mentor behavior
- 2. Collections
- 3. Objects and properties
- 4. Vector embeddings
- 5. BM25 keyword search
- 6. Vector search
- 7. Hybrid search
- 8. RAG with Weaviate
- 9. Chunking
- Introduction


In [5]:
dataset_ref = bigquery.Dataset(
    f"{PROJECT_ID}.{VOICE_DATASET_ID}"
)

dataset_ref.location = BIGQUERY_LOCATION

try:
    dataset = bigquery_client.create_dataset(dataset_ref)
    print("Created dataset:", dataset.full_dataset_id)

except Conflict:
    dataset = bigquery_client.get_dataset(dataset_ref)
    print("Dataset already exists:", dataset.full_dataset_id)

voice_asset_table_ref = (
    f"{PROJECT_ID}.{VOICE_DATASET_ID}.{VOICE_ASSET_TABLE_ID}"
)

voice_test_table_ref = (
    f"{PROJECT_ID}.{VOICE_DATASET_ID}.{VOICE_TEST_TABLE_ID}"
)

print("Voice asset table:", voice_asset_table_ref)
print("Voice test table:", voice_test_table_ref)

Created dataset: leafy-guide-497515-m4:weaviate_voice_widget_cloud_run
Voice asset table: leafy-guide-497515-m4.weaviate_voice_widget_cloud_run.voice_widget_assets
Voice test table: leafy-guide-497515-m4.weaviate_voice_widget_cloud_run.voice_widget_test_requests


In [6]:
def ensure_bigquery_table(
    table_ref: str,
    schema: list[bigquery.SchemaField],
) -> bigquery.Table:
    try:
        table = bigquery_client.get_table(table_ref)

        print("Table already exists:", table.full_table_id)

        return table

    except NotFound:
        print("Table does not exist. Creating:", table_ref)

        table = bigquery.Table(
            table_ref,
            schema=schema,
        )

        created_table = bigquery_client.create_table(table)

        print("Created table:", created_table.full_table_id)

        return created_table

In [7]:
voice_asset_schema = [
    bigquery.SchemaField("asset_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("run_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("asset_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("filename", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("gcs_uri", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("content_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

voice_test_schema = [
    bigquery.SchemaField("test_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("run_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("test_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("request_method", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("endpoint_path", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("example_payload_json", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("expected_result", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("created_at", "TIMESTAMP", mode="REQUIRED"),
]

In [8]:
voice_asset_table = ensure_bigquery_table(
    voice_asset_table_ref,
    voice_asset_schema,
)

voice_test_table = ensure_bigquery_table(
    voice_test_table_ref,
    voice_test_schema,
)

Table does not exist. Creating: leafy-guide-497515-m4.weaviate_voice_widget_cloud_run.voice_widget_assets
Created table: leafy-guide-497515-m4:weaviate_voice_widget_cloud_run.voice_widget_assets
Table does not exist. Creating: leafy-guide-497515-m4.weaviate_voice_widget_cloud_run.voice_widget_test_requests
Created table: leafy-guide-497515-m4:weaviate_voice_widget_cloud_run.voice_widget_test_requests


In [9]:
def gcs_uri_from_blob_name(
    bucket_name: str,
    blob_name: str,
) -> str:
    return f"gs://{bucket_name}/{blob_name}"


def upload_text_to_gcs(
    text: str,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        text,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )


def upload_bytes_to_gcs(
    data: bytes,
    *,
    blob_name: str,
    content_type: str,
) -> str:
    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        data,
        content_type=content_type,
    )

    return gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )


def rows_to_ndjson(
    rows: list[dict],
) -> str:
    return "\n".join(
        json.dumps(
            row,
            ensure_ascii=False,
            default=str,
        )
        for row in rows
    )

In [10]:
def batch_load_rows_to_bigquery(
    *,
    rows: list[dict],
    table_ref: str,
    schema: list[bigquery.SchemaField],
    table_name: str,
    run_id: str,
) -> dict:
    ndjson_text = rows_to_ndjson(rows)

    blob_name = (
        f"{GCS_DEPLOY_PREFIX}/"
        f"{run_id}/"
        f"{table_name}.ndjson"
    )

    gcs_uri = upload_text_to_gcs(
        ndjson_text,
        blob_name=blob_name,
        content_type="application/x-ndjson",
    )

    job_config = bigquery.LoadJobConfig(
        schema=schema,
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )

    started_at = time.perf_counter()

    load_job = bigquery_client.load_table_from_uri(
        gcs_uri,
        table_ref,
        location=dataset.location,
        job_config=job_config,
    )

    load_job.result()

    load_seconds = time.perf_counter() - started_at

    destination_table = bigquery_client.get_table(table_ref)

    print("=" * 100)
    print("Loaded table:", table_name)
    print("Rows:", destination_table.num_rows)
    print("GCS URI:", gcs_uri)
    print("Seconds:", round(load_seconds, 4))

    return {
        "table_name": table_name,
        "gcs_uri": gcs_uri,
        "row_count": destination_table.num_rows,
        "load_seconds": round(load_seconds, 4),
        "job_id": load_job.job_id,
    }

In [11]:
run_id = str(uuid4())
run_timestamp = datetime.now(timezone.utc)

image_uri = (
    f"{LOCATION}-docker.pkg.dev/"
    f"{PROJECT_ID}/"
    f"{ARTIFACT_REPOSITORY}/"
    f"{SERVICE_NAME}:latest"
)

print("Run ID:", run_id)
print("Run timestamp:", run_timestamp.isoformat())
print("Container image URI:")
print(image_uri)

Run ID: df9c8696-dcd5-40f7-ad0e-bb71eb665468
Run timestamp: 2026-07-21T13:14:41.976885+00:00
Container image URI:
us-central1-docker.pkg.dev/leafy-guide-497515-m4/voice-mentor/weaviate-voice-mentor-api:latest


In [12]:
backend_main_py = r'''
from __future__ import annotations

import base64
import os
import time
from typing import Any

import vertexai
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from google import genai
from google.api_core.exceptions import ResourceExhausted
from google.cloud import bigquery
from google.cloud import speech
from google.cloud import texttospeech
from pydantic import BaseModel, Field
from vertexai.language_models import TextEmbeddingModel


PROJECT_ID = os.environ.get("PROJECT_ID", "__PROJECT_ID__")
LOCATION = os.environ.get("LOCATION", "__LOCATION__")
BIGQUERY_LOCATION = os.environ.get("BIGQUERY_LOCATION", "__BIGQUERY_LOCATION__")

DATASET_ID = os.environ.get("DATASET_ID", "__SOURCE_RAG_DATASET_ID__")
CHUNK_TABLE_ID = os.environ.get("CHUNK_TABLE_ID", "__SOURCE_CHUNK_TABLE_ID__")

TEXT_MODEL = os.environ.get("TEXT_MODEL", "__TEXT_MODEL__")
TEXT_EMBEDDING_MODEL = os.environ.get("TEXT_EMBEDDING_MODEL", "__TEXT_EMBEDDING_MODEL__")

ALLOWED_ORIGIN = os.environ.get("ALLOWED_ORIGIN", "__ALLOWED_ORIGIN__")
TOP_K_DEFAULT = int(os.environ.get("TOP_K_DEFAULT", "__TOP_K_DEFAULT__"))

DEFAULT_LANGUAGE_CODE = os.environ.get("DEFAULT_LANGUAGE_CODE", "__DEFAULT_LANGUAGE_CODE__")
DEFAULT_ALTERNATIVE_LANGUAGE_CODE = os.environ.get(
    "DEFAULT_ALTERNATIVE_LANGUAGE_CODE",
    "__DEFAULT_ALTERNATIVE_LANGUAGE_CODE__",
)

MAX_RETRIES = int(os.environ.get("MAX_RETRIES", "5"))
BACKOFF_BASE_SECONDS = int(os.environ.get("BACKOFF_BASE_SECONDS", "8"))
BACKOFF_MAX_SECONDS = int(os.environ.get("BACKOFF_MAX_SECONDS", "90"))

MAX_TTS_CHARS = int(os.environ.get("MAX_TTS_CHARS", "4500"))

CHUNK_TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{CHUNK_TABLE_ID}"

QUERY_EMBEDDING_CACHE: dict[str, list[float]] = {}


vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

genai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

bigquery_client = bigquery.Client(
    project=PROJECT_ID,
)

embedding_model = TextEmbeddingModel.from_pretrained(
    TEXT_EMBEDDING_MODEL,
)

speech_client = speech.SpeechClient()
tts_client = texttospeech.TextToSpeechClient()


class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=4000)
    top_k: int = Field(default=TOP_K_DEFAULT, ge=1, le=20)


class TtsRequest(BaseModel):
    text: str = Field(min_length=1, max_length=6000)
    language_code: str = Field(default=DEFAULT_LANGUAGE_CODE, min_length=2, max_length=16)


class VoiceChatRequest(BaseModel):
    audio_base64: str = Field(min_length=1)
    mime_type: str = Field(default="audio/webm;codecs=opus")
    language_code: str = Field(default=DEFAULT_LANGUAGE_CODE, min_length=2, max_length=16)
    alternative_language_code: str = Field(
        default=DEFAULT_ALTERNATIVE_LANGUAGE_CODE,
        min_length=2,
        max_length=16,
    )
    top_k: int = Field(default=TOP_K_DEFAULT, ge=1, le=20)


class SourceChunk(BaseModel):
    chunk_id: str
    source_title: str
    chunk_number: int
    distance: float
    preview: str


class ChatResponse(BaseModel):
    answer: str
    sources: list[SourceChunk]


class TtsResponse(BaseModel):
    audio_base64: str
    mime_type: str
    language_code: str


class VoiceChatResponse(BaseModel):
    transcript: str
    answer: str
    audio_base64: str
    audio_mime_type: str
    sources: list[SourceChunk]


app = FastAPI(
    title="Weaviate Voice Mentor API",
    version="2.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        ALLOWED_ORIGIN,
        "http://localhost:3000",
        "http://localhost:5173",
        "http://127.0.0.1:5173",
    ],
    allow_credentials=False,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["*"],
)


def is_rate_limit_error(exc: Exception) -> bool:
    message = str(exc).lower()

    return (
        isinstance(exc, ResourceExhausted)
        or "resource exhausted" in message
        or "429" in message
        or "quota" in message
        or "rate limit" in message
    )


def sleep_with_backoff(
    attempt: int,
) -> None:
    sleep_seconds = min(
        BACKOFF_MAX_SECONDS,
        BACKOFF_BASE_SECONDS * (2 ** (attempt - 1)),
    )

    print(
        f"Rate limit detected. Waiting {sleep_seconds} seconds before retry..."
    )

    time.sleep(sleep_seconds)


def shorten_text_for_embedding(
    text: str,
    max_chars: int = 3000,
) -> str:
    clean_text = " ".join(text.split())

    if len(clean_text) <= max_chars:
        return clean_text

    return clean_text[:max_chars].rsplit(" ", 1)[0]


def get_query_embedding(
    query: str,
) -> list[float]:
    clean_query = shorten_text_for_embedding(query)

    if clean_query in QUERY_EMBEDDING_CACHE:
        return QUERY_EMBEDDING_CACHE[clean_query]

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            embeddings = embedding_model.get_embeddings(
                [clean_query]
            )

            embedding = list(embeddings[0].values)

            QUERY_EMBEDDING_CACHE[clean_query] = embedding

            return embedding

        except Exception as exc:
            if is_rate_limit_error(exc) and attempt < MAX_RETRIES:
                sleep_with_backoff(attempt)
                continue

            raise

    raise RuntimeError("Could not create query embedding.")


def search_chunks(
    query: str,
    top_k: int,
) -> list[dict[str, Any]]:
    query_embedding = get_query_embedding(query)

    sql = f"""
    SELECT
      base.chunk_id AS chunk_id,
      base.document_id AS document_id,
      base.document_number AS document_number,
      base.chunk_number AS chunk_number,
      base.global_chunk_number AS global_chunk_number,
      base.source_type AS source_type,
      base.source_title AS source_title,
      base.section_title AS section_title,
      base.chunk_text AS chunk_text,
      distance
    FROM VECTOR_SEARCH(
      (
        SELECT *
        FROM `{CHUNK_TABLE_REF}`
      ),
      'embedding',
      (
        SELECT
          @query_embedding AS embedding
      ),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance ASC
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "query_embedding",
                "FLOAT64",
                query_embedding,
            ),
            bigquery.ScalarQueryParameter(
                "top_k",
                "INT64",
                top_k,
            ),
        ]
    )

    query_job = bigquery_client.query(
        sql,
        job_config=job_config,
        location=BIGQUERY_LOCATION,
    )

    return [
        {
            "chunk_id": row.chunk_id,
            "document_id": row.document_id,
            "document_number": row.document_number,
            "chunk_number": row.chunk_number,
            "global_chunk_number": row.global_chunk_number,
            "source_type": row.source_type,
            "source_title": row.source_title,
            "section_title": row.section_title,
            "chunk_text": row.chunk_text,
            "distance": row.distance,
        }
        for row in query_job
    ]


def source_chunks_to_response(
    chunks: list[dict[str, Any]],
) -> list[SourceChunk]:
    return [
        SourceChunk(
            chunk_id=chunk["chunk_id"],
            source_title=chunk["source_title"],
            chunk_number=int(chunk["global_chunk_number"]),
            distance=float(chunk["distance"]),
            preview=chunk["chunk_text"][:260],
        )
        for chunk in chunks
    ]


def build_rag_context(
    chunks: list[dict[str, Any]],
) -> str:
    parts = []

    for index, chunk in enumerate(chunks, start=1):
        parts.append(
            "\n".join(
                [
                    f"[SOURCE {index}]",
                    f"Chunk ID: {chunk['chunk_id']}",
                    f"Section: {chunk['source_title']}",
                    f"Distance: {chunk['distance']:.4f}",
                    f"Text: {chunk['chunk_text']}",
                ]
            )
        )

    return "\n\n---\n\n".join(parts)


def generate_answer(
    question: str,
    chunks: list[dict[str, Any]],
) -> str:
    context = build_rag_context(chunks)

    prompt = f"""
You are Weaviate Voice Mentor.

Answer in Polish by default.

You help a Python backend learner understand Weaviate, vector databases,
embeddings, semantic search, BM25, hybrid search, RAG, collections,
objects, references, chunks, and Weaviate Cloud.

Use only the retrieved context.

User question:
{question}

Retrieved context:
{context}

Answer style:
- explain simply
- use practical examples
- use SQL table/row analogies when helpful
- mention SOURCE numbers when using evidence
- do not invent information outside the context
- if the context is insufficient, say what is missing
- avoid long theory unless the user asks for it
- keep the answer good for voice playback
"""

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = genai_client.models.generate_content(
                model=TEXT_MODEL,
                contents=prompt,
            )

            return response.text

        except Exception as exc:
            if is_rate_limit_error(exc) and attempt < MAX_RETRIES:
                sleep_with_backoff(attempt)
                continue

            raise

    raise RuntimeError("Could not generate answer.")


def synthesize_speech_base64(
    text: str,
    language_code: str,
) -> str:
    safe_text = text.strip()[:MAX_TTS_CHARS]

    synthesis_input = texttospeech.SynthesisInput(
        text=safe_text,
    )

    voice = texttospeech.VoiceSelectionParams(
        language_code=language_code,
        ssml_gender=texttospeech.SsmlVoiceGender.NEUTRAL,
    )

    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3,
        speaking_rate=1.0,
        pitch=0.0,
    )

    response = tts_client.synthesize_speech(
        input=synthesis_input,
        voice=voice,
        audio_config=audio_config,
    )

    return base64.b64encode(response.audio_content).decode("utf-8")


def transcribe_audio_base64(
    audio_base64: str,
    language_code: str,
    alternative_language_code: str,
) -> str:
    audio_bytes = base64.b64decode(audio_base64)

    audio = speech.RecognitionAudio(
        content=audio_bytes,
    )

    config = speech.RecognitionConfig(
        encoding=speech.RecognitionConfig.AudioEncoding.WEBM_OPUS,
        sample_rate_hertz=48000,
        language_code=language_code,
        alternative_language_codes=[
            alternative_language_code,
        ],
        enable_automatic_punctuation=True,
    )

    response = speech_client.recognize(
        config=config,
        audio=audio,
    )

    transcript_parts = []

    for result in response.results:
        if result.alternatives:
            transcript_parts.append(
                result.alternatives[0].transcript
            )

    transcript = " ".join(transcript_parts).strip()

    if not transcript:
        raise ValueError("Speech-to-Text returned empty transcript.")

    return transcript


@app.get("/")
def root() -> dict[str, str]:
    return {
        "status": "ok",
        "service": "Weaviate Voice Mentor API",
        "version": "2.0.0",
    }


@app.get("/health")
def health() -> dict[str, str]:
    return {
        "status": "healthy",
        "project_id": PROJECT_ID,
        "dataset_id": DATASET_ID,
        "chunk_table_id": CHUNK_TABLE_ID,
        "bigquery_location": BIGQUERY_LOCATION,
        "text_model": TEXT_MODEL,
        "embedding_model": TEXT_EMBEDDING_MODEL,
        "default_language_code": DEFAULT_LANGUAGE_CODE,
    }


@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest) -> ChatResponse:
    try:
        chunks = search_chunks(
            request.message,
            top_k=request.top_k,
        )

        answer = generate_answer(
            request.message,
            chunks,
        )

        return ChatResponse(
            answer=answer,
            sources=source_chunks_to_response(chunks),
        )

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Chat request failed: {exc}",
        ) from exc


@app.post("/tts", response_model=TtsResponse)
def tts(request: TtsRequest) -> TtsResponse:
    try:
        audio_base64 = synthesize_speech_base64(
            text=request.text,
            language_code=request.language_code,
        )

        return TtsResponse(
            audio_base64=audio_base64,
            mime_type="audio/mpeg",
            language_code=request.language_code,
        )

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"TTS request failed: {exc}",
        ) from exc


@app.post("/voice-chat", response_model=VoiceChatResponse)
def voice_chat(request: VoiceChatRequest) -> VoiceChatResponse:
    try:
        transcript = transcribe_audio_base64(
            audio_base64=request.audio_base64,
            language_code=request.language_code,
            alternative_language_code=request.alternative_language_code,
        )

        chunks = search_chunks(
            transcript,
            top_k=request.top_k,
        )

        answer = generate_answer(
            transcript,
            chunks,
        )

        audio_base64 = synthesize_speech_base64(
            text=answer,
            language_code=request.language_code,
        )

        return VoiceChatResponse(
            transcript=transcript,
            answer=answer,
            audio_base64=audio_base64,
            audio_mime_type="audio/mpeg",
            sources=source_chunks_to_response(chunks),
        )

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Voice chat request failed: {exc}",
        ) from exc
'''.strip()

backend_main_py = (
    backend_main_py
    .replace("__PROJECT_ID__", PROJECT_ID)
    .replace("__LOCATION__", LOCATION)
    .replace("__BIGQUERY_LOCATION__", source_table.location)
    .replace("__SOURCE_RAG_DATASET_ID__", SOURCE_RAG_DATASET_ID)
    .replace("__SOURCE_CHUNK_TABLE_ID__", SOURCE_CHUNK_TABLE_ID)
    .replace("__TEXT_MODEL__", TEXT_MODEL)
    .replace("__TEXT_EMBEDDING_MODEL__", TEXT_EMBEDDING_MODEL)
    .replace("__ALLOWED_ORIGIN__", ALLOWED_ORIGIN)
    .replace("__TOP_K_DEFAULT__", str(TOP_K_DEFAULT))
    .replace("__DEFAULT_LANGUAGE_CODE__", DEFAULT_LANGUAGE_CODE)
    .replace("__DEFAULT_ALTERNATIVE_LANGUAGE_CODE__", DEFAULT_ALTERNATIVE_LANGUAGE_CODE)
)

print(backend_main_py[:5000])

from __future__ import annotations

import base64
import os
import time
from typing import Any

import vertexai
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from google import genai
from google.api_core.exceptions import ResourceExhausted
from google.cloud import bigquery
from google.cloud import speech
from google.cloud import texttospeech
from pydantic import BaseModel, Field
from vertexai.language_models import TextEmbeddingModel


PROJECT_ID = os.environ.get("PROJECT_ID", "leafy-guide-497515-m4")
LOCATION = os.environ.get("LOCATION", "us-central1")
BIGQUERY_LOCATION = os.environ.get("BIGQUERY_LOCATION", "EU")

DATASET_ID = os.environ.get("DATASET_ID", "weaviate_mentor_rag_backend")
CHUNK_TABLE_ID = os.environ.get("CHUNK_TABLE_ID", "weaviate_mentor_chunks")

TEXT_MODEL = os.environ.get("TEXT_MODEL", "gemini-2.5-flash")
TEXT_EMBEDDING_MODEL = os.environ.get("TEXT_EMBEDDING_MODEL", "text-embedding-005")

ALLOWED_ORIGIN = os.environ.get(

In [13]:
backend_requirements_txt = """
fastapi>=0.115.0
uvicorn[standard]>=0.30.0
google-cloud-bigquery>=3.25.0
google-cloud-aiplatform>=1.70.0
google-genai>=1.0.0
google-cloud-texttospeech>=2.18.0
google-cloud-speech>=2.28.0
pydantic>=2.8.0
""".strip()

print(backend_requirements_txt)

fastapi>=0.115.0
uvicorn[standard]>=0.30.0
google-cloud-bigquery>=3.25.0
google-cloud-aiplatform>=1.70.0
google-genai>=1.0.0
google-cloud-texttospeech>=2.18.0
google-cloud-speech>=2.28.0
pydantic>=2.8.0


In [14]:
backend_dockerfile = """
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .

EXPOSE 8080

CMD ["sh", "-c", "uvicorn main:app --host 0.0.0.0 --port ${PORT:-8080}"]
""".strip()

print(backend_dockerfile)

FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .

EXPOSE 8080

CMD ["sh", "-c", "uvicorn main:app --host 0.0.0.0 --port ${PORT:-8080}"]


In [15]:
deploy_sh = f"""
#!/usr/bin/env bash
set -euo pipefail

PROJECT_ID="{PROJECT_ID}"
REGION="{LOCATION}"
SERVICE_NAME="{SERVICE_NAME}"
ARTIFACT_REPOSITORY="{ARTIFACT_REPOSITORY}"

IMAGE_URI="{image_uri}"

gcloud config set project "$PROJECT_ID"

gcloud services enable \\
  run.googleapis.com \\
  cloudbuild.googleapis.com \\
  artifactregistry.googleapis.com \\
  aiplatform.googleapis.com \\
  bigquery.googleapis.com \\
  speech.googleapis.com \\
  texttospeech.googleapis.com

gcloud artifacts repositories describe "$ARTIFACT_REPOSITORY" \\
  --location "$REGION" \\
  >/dev/null 2>&1 \\
  || gcloud artifacts repositories create "$ARTIFACT_REPOSITORY" \\
      --repository-format=docker \\
      --location "$REGION" \\
      --description="Docker repository for Weaviate Voice Mentor"

gcloud builds submit \\
  --tag "$IMAGE_URI" \\
  .

gcloud run deploy "$SERVICE_NAME" \\
  --image "$IMAGE_URI" \\
  --region "$REGION" \\
  --platform managed \\
  --allow-unauthenticated \\
  --memory 2Gi \\
  --cpu 2 \\
  --timeout 300 \\
  --max-instances 3 \\
  --set-env-vars PROJECT_ID={PROJECT_ID},LOCATION={LOCATION},BIGQUERY_LOCATION={source_table.location},DATASET_ID={SOURCE_RAG_DATASET_ID},CHUNK_TABLE_ID={SOURCE_CHUNK_TABLE_ID},TEXT_MODEL={TEXT_MODEL},TEXT_EMBEDDING_MODEL={TEXT_EMBEDDING_MODEL},ALLOWED_ORIGIN={ALLOWED_ORIGIN},TOP_K_DEFAULT={TOP_K_DEFAULT},DEFAULT_LANGUAGE_CODE={DEFAULT_LANGUAGE_CODE},DEFAULT_ALTERNATIVE_LANGUAGE_CODE={DEFAULT_ALTERNATIVE_LANGUAGE_CODE}

SERVICE_URL="$(gcloud run services describe "$SERVICE_NAME" --region "$REGION" --format='value(status.url)')"

echo ""
echo "Cloud Run service URL:"
echo "$SERVICE_URL"
echo ""
echo "Health check:"
echo "$SERVICE_URL/health"
""".strip()

print(deploy_sh)

#!/usr/bin/env bash
set -euo pipefail

PROJECT_ID="leafy-guide-497515-m4"
REGION="us-central1"
SERVICE_NAME="weaviate-voice-mentor-api"
ARTIFACT_REPOSITORY="voice-mentor"

IMAGE_URI="us-central1-docker.pkg.dev/leafy-guide-497515-m4/voice-mentor/weaviate-voice-mentor-api:latest"

gcloud config set project "$PROJECT_ID"

gcloud services enable \
  run.googleapis.com \
  cloudbuild.googleapis.com \
  artifactregistry.googleapis.com \
  aiplatform.googleapis.com \
  bigquery.googleapis.com \
  speech.googleapis.com \
  texttospeech.googleapis.com

gcloud artifacts repositories describe "$ARTIFACT_REPOSITORY" \
  --location "$REGION" \
  >/dev/null 2>&1 \
  || gcloud artifacts repositories create "$ARTIFACT_REPOSITORY" \
      --repository-format=docker \
      --location "$REGION" \
      --description="Docker repository for Weaviate Voice Mentor"

gcloud builds submit \
  --tag "$IMAGE_URI" \
  .

gcloud run deploy "$SERVICE_NAME" \
  --image "$IMAGE_URI" \
  --region "$REGION" \
  --plat

In [16]:
smoke_test_sh = """
#!/usr/bin/env bash
set -euo pipefail

SERVICE_URL="${1:-}"

if [ -z "$SERVICE_URL" ]; then
  echo "Usage: bash smoke_test.sh https://your-cloud-run-url"
  exit 1
fi

echo "Health check"
curl -s "$SERVICE_URL/health" | python -m json.tool

echo ""
echo "Text chat test"
curl -s -X POST "$SERVICE_URL/chat" \\
  -H "Content-Type: application/json" \\
  -d '{
    "message": "Co to jest kolekcja w Weaviate i jak to porównać do tabeli SQL?",
    "top_k": 5
  }' | python -m json.tool

echo ""
echo "TTS test"
curl -s -X POST "$SERVICE_URL/tts" \\
  -H "Content-Type: application/json" \\
  -d '{
    "text": "Cześć, jestem Weaviate Voice Mentor. Możesz zapytać mnie o kolekcje, embeddingi, RAG i hybrid search.",
    "language_code": "pl-PL"
  }' | python -m json.tool
""".strip()

print(smoke_test_sh)

#!/usr/bin/env bash
set -euo pipefail

SERVICE_URL="${1:-}"

if [ -z "$SERVICE_URL" ]; then
  echo "Usage: bash smoke_test.sh https://your-cloud-run-url"
  exit 1
fi

echo "Health check"
curl -s "$SERVICE_URL/health" | python -m json.tool

echo ""
echo "Text chat test"
curl -s -X POST "$SERVICE_URL/chat" \
  -H "Content-Type: application/json" \
  -d '{
    "message": "Co to jest kolekcja w Weaviate i jak to porównać do tabeli SQL?",
    "top_k": 5
  }' | python -m json.tool

echo ""
echo "TTS test"
curl -s -X POST "$SERVICE_URL/tts" \
  -H "Content-Type: application/json" \
  -d '{
    "text": "Cześć, jestem Weaviate Voice Mentor. Możesz zapytać mnie o kolekcje, embeddingi, RAG i hybrid search.",
    "language_code": "pl-PL"
  }' | python -m json.tool


In [17]:
widget_js = r'''
(function () {
  const API_URL = window.WEAVIATE_MENTOR_API_URL || "PASTE_CLOUD_RUN_URL_HERE";

  const root = document.getElementById("weaviate-voice-mentor-widget");

  if (!root) {
    console.error("Missing #weaviate-voice-mentor-widget container.");
    return;
  }

  root.innerHTML = `
    <button id="wvm-open-button">Ask Weaviate Mentor</button>

    <div id="wvm-panel" style="display:none;">
      <div id="wvm-header">
        <div>
          <strong>Weaviate Voice Mentor</strong>
          <div id="wvm-subtitle">RAG + Gemini + Google Cloud Voice</div>
        </div>
        <button id="wvm-close-button">×</button>
      </div>

      <div id="wvm-messages"></div>

      <div id="wvm-controls">
        <textarea id="wvm-input" placeholder="Zapytaj o Weaviate..."></textarea>

        <div id="wvm-buttons">
          <button id="wvm-send-button">Send</button>
          <button id="wvm-browser-mic-button" title="Browser speech recognition">🎙️ Browser</button>
          <button id="wvm-cloud-record-button" title="Cloud Speech-to-Text voice chat">● Cloud Voice</button>
          <button id="wvm-speak-button" title="Play last answer with Cloud TTS">🔊 Speak</button>
        </div>
      </div>
    </div>
  `;

  const style = document.createElement("style");
  style.textContent = `
    #weaviate-voice-mentor-widget {
      position: fixed;
      right: 24px;
      bottom: 24px;
      z-index: 9999;
      font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    }

    #wvm-open-button {
      border: none;
      border-radius: 999px;
      padding: 14px 18px;
      background: #111827;
      color: white;
      box-shadow: 0 12px 28px rgba(0,0,0,0.24);
      cursor: pointer;
      font-weight: 650;
    }

    #wvm-panel {
      width: 390px;
      height: 620px;
      background: #0f172a;
      color: white;
      border-radius: 18px;
      overflow: hidden;
      box-shadow: 0 22px 60px rgba(0,0,0,0.35);
      border: 1px solid rgba(255,255,255,0.12);
    }

    #wvm-header {
      height: 64px;
      padding: 0 14px;
      display: flex;
      align-items: center;
      justify-content: space-between;
      background: #111827;
      border-bottom: 1px solid rgba(255,255,255,0.12);
    }

    #wvm-subtitle {
      margin-top: 3px;
      font-size: 12px;
      color: #9ca3af;
    }

    #wvm-close-button {
      background: transparent;
      color: white;
      border: none;
      font-size: 26px;
      cursor: pointer;
    }

    #wvm-messages {
      height: 430px;
      overflow-y: auto;
      padding: 14px;
      display: flex;
      flex-direction: column;
      gap: 10px;
    }

    .wvm-message {
      padding: 10px 12px;
      border-radius: 12px;
      line-height: 1.45;
      font-size: 14px;
      white-space: pre-wrap;
      max-width: 92%;
    }

    .wvm-user {
      align-self: flex-end;
      background: #2563eb;
    }

    .wvm-bot {
      align-self: flex-start;
      background: #1f2937;
    }

    .wvm-system {
      align-self: center;
      background: #334155;
      font-size: 12px;
      color: #e5e7eb;
    }

    #wvm-controls {
      padding: 12px;
      background: #111827;
      border-top: 1px solid rgba(255,255,255,0.12);
    }

    #wvm-input {
      width: 100%;
      box-sizing: border-box;
      resize: none;
      height: 64px;
      border-radius: 10px;
      border: 1px solid rgba(255,255,255,0.16);
      background: #020617;
      color: white;
      padding: 9px;
      outline: none;
    }

    #wvm-buttons {
      display: grid;
      grid-template-columns: 1fr 1fr 1fr 1fr;
      gap: 6px;
      margin-top: 8px;
    }

    #wvm-buttons button {
      border: none;
      border-radius: 10px;
      padding: 8px 6px;
      background: #334155;
      color: white;
      cursor: pointer;
      font-size: 12px;
    }

    #wvm-cloud-record-button.recording {
      background: #dc2626;
    }
  `;

  document.head.appendChild(style);

  const openButton = document.getElementById("wvm-open-button");
  const closeButton = document.getElementById("wvm-close-button");
  const panel = document.getElementById("wvm-panel");
  const messages = document.getElementById("wvm-messages");
  const input = document.getElementById("wvm-input");
  const sendButton = document.getElementById("wvm-send-button");
  const browserMicButton = document.getElementById("wvm-browser-mic-button");
  const cloudRecordButton = document.getElementById("wvm-cloud-record-button");
  const speakButton = document.getElementById("wvm-speak-button");

  let lastAnswer = "";
  let mediaRecorder = null;
  let recordedChunks = [];
  let isRecording = false;

  function addMessage(text, role) {
    const div = document.createElement("div");
    div.className = "wvm-message " + (
      role === "user"
        ? "wvm-user"
        : role === "system"
          ? "wvm-system"
          : "wvm-bot"
    );

    div.textContent = text;
    messages.appendChild(div);
    messages.scrollTop = messages.scrollHeight;

    return div;
  }

  function base64FromBlob(blob) {
    return new Promise((resolve, reject) => {
      const reader = new FileReader();

      reader.onloadend = function () {
        const dataUrl = reader.result;
        const base64 = String(dataUrl).split(",")[1];
        resolve(base64);
      };

      reader.onerror = reject;
      reader.readAsDataURL(blob);
    });
  }

  function playBase64Audio(audioBase64, mimeType) {
    const audio = new Audio("data:" + mimeType + ";base64," + audioBase64);
    audio.play();
  }

  async function sendTextChat() {
    const message = input.value.trim();

    if (!message) {
      return;
    }

    input.value = "";

    addMessage(message, "user");
    const pending = addMessage("Myślę...", "bot");

    try {
      const response = await fetch(API_URL + "/chat", {
        method: "POST",
        headers: {
          "Content-Type": "application/json"
        },
        body: JSON.stringify({
          message: message,
          top_k: 8
        })
      });

      if (!response.ok) {
        throw new Error("HTTP " + response.status);
      }

      const data = await response.json();

      pending.textContent = data.answer;
      lastAnswer = data.answer;
    } catch (error) {
      pending.textContent = "Błąd połączenia z Weaviate Mentor API: " + error.message;
    }
  }

  async function speakLastAnswerWithCloudTts() {
    if (!lastAnswer) {
      addMessage("Nie ma jeszcze odpowiedzi do odtworzenia.", "system");
      return;
    }

    try {
      const response = await fetch(API_URL + "/tts", {
        method: "POST",
        headers: {
          "Content-Type": "application/json"
        },
        body: JSON.stringify({
          text: lastAnswer,
          language_code: "pl-PL"
        })
      });

      if (!response.ok) {
        throw new Error("HTTP " + response.status);
      }

      const data = await response.json();

      playBase64Audio(data.audio_base64, data.mime_type);
    } catch (error) {
      addMessage("TTS error: " + error.message, "system");
    }
  }

  function startBrowserSpeechRecognition() {
    const SpeechRecognition =
      window.SpeechRecognition || window.webkitSpeechRecognition;

    if (!SpeechRecognition) {
      addMessage("Ta przeglądarka nie wspiera Web Speech API.", "system");
      return;
    }

    const recognition = new SpeechRecognition();
    recognition.lang = "pl-PL";
    recognition.interimResults = false;
    recognition.maxAlternatives = 1;

    recognition.onstart = function () {
      addMessage("Słucham przez przeglądarkę...", "system");
    };

    recognition.onresult = function (event) {
      input.value = event.results[0][0].transcript;
    };

    recognition.onerror = function (event) {
      addMessage("Browser speech recognition error: " + event.error, "system");
    };

    recognition.start();
  }

  async function toggleCloudVoiceRecording() {
    if (isRecording && mediaRecorder) {
      mediaRecorder.stop();
      return;
    }

    if (!navigator.mediaDevices || !navigator.mediaDevices.getUserMedia) {
      addMessage("Ta przeglądarka nie wspiera nagrywania mikrofonu.", "system");
      return;
    }

    try {
      const stream = await navigator.mediaDevices.getUserMedia({
        audio: true
      });

      recordedChunks = [];

      const mimeType = MediaRecorder.isTypeSupported("audio/webm;codecs=opus")
        ? "audio/webm;codecs=opus"
        : "audio/webm";

      mediaRecorder = new MediaRecorder(stream, {
        mimeType: mimeType
      });

      mediaRecorder.ondataavailable = function (event) {
        if (event.data.size > 0) {
          recordedChunks.push(event.data);
        }
      };

      mediaRecorder.onstart = function () {
        isRecording = true;
        cloudRecordButton.classList.add("recording");
        cloudRecordButton.textContent = "Stop";
        addMessage("Nagrywam pytanie...", "system");
      };

      mediaRecorder.onstop = async function () {
        isRecording = false;
        cloudRecordButton.classList.remove("recording");
        cloudRecordButton.textContent = "● Cloud Voice";

        stream.getTracks().forEach(track => track.stop());

        const audioBlob = new Blob(recordedChunks, {
          type: mimeType
        });

        const audioBase64 = await base64FromBlob(audioBlob);

        addMessage("Wysyłam audio do Google Speech-to-Text...", "system");
        const pending = addMessage("Transkrypcja i odpowiedź w toku...", "bot");

        try {
          const response = await fetch(API_URL + "/voice-chat", {
            method: "POST",
            headers: {
              "Content-Type": "application/json"
            },
            body: JSON.stringify({
              audio_base64: audioBase64,
              mime_type: mimeType,
              language_code: "pl-PL",
              alternative_language_code: "en-US",
              top_k: 8
            })
          });

          if (!response.ok) {
            throw new Error("HTTP " + response.status);
          }

          const data = await response.json();

          addMessage("Transkrypcja: " + data.transcript, "user");

          pending.textContent = data.answer;
          lastAnswer = data.answer;

          playBase64Audio(data.audio_base64, data.audio_mime_type);
        } catch (error) {
          pending.textContent = "Voice chat error: " + error.message;
        }
      };

      mediaRecorder.start();
    } catch (error) {
      addMessage("Microphone error: " + error.message, "system");
    }
  }

  openButton.addEventListener("click", function () {
    panel.style.display = "block";
    openButton.style.display = "none";
  });

  closeButton.addEventListener("click", function () {
    panel.style.display = "none";
    openButton.style.display = "block";
  });

  sendButton.addEventListener("click", sendTextChat);
  speakButton.addEventListener("click", speakLastAnswerWithCloudTts);
  browserMicButton.addEventListener("click", startBrowserSpeechRecognition);
  cloudRecordButton.addEventListener("click", toggleCloudVoiceRecording);

  input.addEventListener("keydown", function (event) {
    if (event.key === "Enter" && !event.shiftKey) {
      event.preventDefault();
      sendTextChat();
    }
  });
})();
'''.strip()

print(widget_js[:5000])

(function () {
  const API_URL = window.WEAVIATE_MENTOR_API_URL || "PASTE_CLOUD_RUN_URL_HERE";

  const root = document.getElementById("weaviate-voice-mentor-widget");

  if (!root) {
    console.error("Missing #weaviate-voice-mentor-widget container.");
    return;
  }

  root.innerHTML = `
    <button id="wvm-open-button">Ask Weaviate Mentor</button>

    <div id="wvm-panel" style="display:none;">
      <div id="wvm-header">
        <div>
          <strong>Weaviate Voice Mentor</strong>
          <div id="wvm-subtitle">RAG + Gemini + Google Cloud Voice</div>
        </div>
        <button id="wvm-close-button">×</button>
      </div>

      <div id="wvm-messages"></div>

      <div id="wvm-controls">
        <textarea id="wvm-input" placeholder="Zapytaj o Weaviate..."></textarea>

        <div id="wvm-buttons">
          <button id="wvm-send-button">Send</button>
          <button id="wvm-browser-mic-button" title="Browser speech recognition">🎙️ Browser</button>
          <button id=

In [18]:
lovable_embed_html = """
<div id="weaviate-voice-mentor-widget"></div>

<script>
  window.WEAVIATE_MENTOR_API_URL = "PASTE_CLOUD_RUN_URL_HERE";
</script>

<script>
__WIDGET_JS__
</script>
""".replace(
    "__WIDGET_JS__",
    widget_js,
).strip()

print(lovable_embed_html[:5000])

<div id="weaviate-voice-mentor-widget"></div>

<script>
  window.WEAVIATE_MENTOR_API_URL = "PASTE_CLOUD_RUN_URL_HERE";
</script>

<script>
(function () {
  const API_URL = window.WEAVIATE_MENTOR_API_URL || "PASTE_CLOUD_RUN_URL_HERE";

  const root = document.getElementById("weaviate-voice-mentor-widget");

  if (!root) {
    console.error("Missing #weaviate-voice-mentor-widget container.");
    return;
  }

  root.innerHTML = `
    <button id="wvm-open-button">Ask Weaviate Mentor</button>

    <div id="wvm-panel" style="display:none;">
      <div id="wvm-header">
        <div>
          <strong>Weaviate Voice Mentor</strong>
          <div id="wvm-subtitle">RAG + Gemini + Google Cloud Voice</div>
        </div>
        <button id="wvm-close-button">×</button>
      </div>

      <div id="wvm-messages"></div>

      <div id="wvm-controls">
        <textarea id="wvm-input" placeholder="Zapytaj o Weaviate..."></textarea>

        <div id="wvm-buttons">
          <button id="wvm-send-butto

In [19]:
widget_html = """
<div id="weaviate-voice-mentor-widget"></div>

<script>
  window.WEAVIATE_MENTOR_API_URL = "PASTE_CLOUD_RUN_URL_HERE";
</script>

<script src="./weaviate-voice-widget.js"></script>
""".strip()

print(widget_html)

<div id="weaviate-voice-mentor-widget"></div>

<script>
  window.WEAVIATE_MENTOR_API_URL = "PASTE_CLOUD_RUN_URL_HERE";
</script>

<script src="./weaviate-voice-widget.js"></script>


In [21]:
readme_md = f"""
# Weaviate Voice Mentor API

This package contains a Cloud Run-ready FastAPI backend and a Lovable widget.

## What this backend does

Endpoints:

- `GET /`
- `GET /health`
- `POST /chat`
- `POST /tts`
- `POST /voice-chat`

## Architecture

```text
Lovable website
        ↓
Voice/text widget
        ↓
Cloud Run FastAPI API
        ↓
Speech-to-Text / Text-to-Speech
        ↓
BigQuery VECTOR_SEARCH over Weaviate Mentor chunks
        ↓
Gemini answer"""

In [23]:
backend_files = {
    "main.py": backend_main_py,
    "requirements.txt": backend_requirements_txt,
    "Dockerfile": backend_dockerfile,
    "deploy.sh": deploy_sh,
    "smoke_test.sh": smoke_test_sh,
    "README.md": readme_md,
    "weaviate-voice-widget.js": widget_js,
    "widget.html": widget_html,
    "lovable_embed.html": lovable_embed_html,
}

zip_buffer = BytesIO()

with zipfile.ZipFile(
    zip_buffer,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zip_file:
    for filename, content in backend_files.items():
        zip_file.writestr(
            filename,
            content,
        )

zip_buffer.seek(0)

package_blob_name = (
    f"{GCS_BACKEND_PREFIX}/"
    f"{run_id}/"
    "weaviate_voice_mentor_cloud_run_package.zip"
)

backend_package_gcs_uri = upload_bytes_to_gcs(
    zip_buffer.getvalue(),
    blob_name=package_blob_name,
    content_type="application/zip",
)

print("Backend package saved to:")
print(backend_package_gcs_uri)
print("Package bytes:", len(zip_buffer.getvalue()))

Backend package saved to:
gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/backend/df9c8696-dcd5-40f7-ad0e-bb71eb665468/weaviate_voice_mentor_cloud_run_package.zip
Package bytes: 13491


In [24]:
voice_asset_rows = []

for filename, content in backend_files.items():
    if filename.endswith(".py"):
        content_type = "text/x-python"
        asset_type = "backend_code"
        description = "FastAPI backend"

    elif filename.endswith(".js"):
        content_type = "application/javascript"
        asset_type = "widget_code"
        description = "Voice widget JavaScript"

    elif filename.endswith(".html"):
        content_type = "text/html"
        asset_type = "widget_code"
        description = "Embeddable Lovable widget"

    elif filename.endswith(".md"):
        content_type = "text/markdown"
        asset_type = "documentation"
        description = "Deployment documentation"

    elif filename.endswith(".sh"):
        content_type = "text/x-shellscript"
        asset_type = "deployment_script"
        description = "Shell deployment or test script"

    elif filename == "Dockerfile":
        content_type = "text/plain"
        asset_type = "deployment_code"
        description = "Cloud Run Dockerfile"

    else:
        content_type = "text/plain"
        asset_type = "other"
        description = "Generated asset"

    blob_name = (
        f"{GCS_BACKEND_PREFIX}/"
        f"{run_id}/"
        f"{filename}"
    )

    gcs_uri = upload_text_to_gcs(
        content,
        blob_name=blob_name,
        content_type=content_type,
    )

    voice_asset_rows.append(
        {
            "asset_id": str(uuid4()),
            "run_id": run_id,
            "asset_type": asset_type,
            "filename": filename,
            "gcs_uri": gcs_uri,
            "content_type": content_type,
            "description": description,
            "created_at": datetime.now(timezone.utc).isoformat(),
        }
    )

voice_asset_rows.append(
    {
        "asset_id": str(uuid4()),
        "run_id": run_id,
        "asset_type": "backend_package",
        "filename": "weaviate_voice_mentor_cloud_run_package.zip",
        "gcs_uri": backend_package_gcs_uri,
        "content_type": "application/zip",
        "description": "Full backend and widget package",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
)

print("Voice asset rows:", len(voice_asset_rows))

pd.DataFrame(
    [
        {
            "asset_type": row["asset_type"],
            "filename": row["filename"],
            "gcs_uri": row["gcs_uri"],
        }
        for row in voice_asset_rows
    ]
)

Voice asset rows: 10


,asset_type,filename,gcs_uri
0,backend_code,main.py,gs://leafy-guide-497515-m4-vector-assets/weavi...
1,other,requirements.txt,gs://leafy-guide-497515-m4-vector-assets/weavi...
2,deployment_code,Dockerfile,gs://leafy-guide-497515-m4-vector-assets/weavi...
3,deployment_script,deploy.sh,gs://leafy-guide-497515-m4-vector-assets/weavi...
4,deployment_script,smoke_test.sh,gs://leafy-guide-497515-m4-vector-assets/weavi...
5,documentation,README.md,gs://leafy-guide-497515-m4-vector-assets/weavi...
6,widget_code,weaviate-voice-widget.js,gs://leafy-guide-497515-m4-vector-assets/weavi...
7,widget_code,widget.html,gs://leafy-guide-497515-m4-vector-assets/weavi...
8,widget_code,lovable_embed.html,gs://leafy-guide-497515-m4-vector-assets/weavi...
9,backend_package,weaviate_voice_mentor_cloud_run_package.zip,gs://leafy-guide-497515-m4-vector-assets/weavi...


In [25]:
asset_load_result = batch_load_rows_to_bigquery(
    rows=voice_asset_rows,
    table_ref=voice_asset_table_ref,
    schema=voice_asset_schema,
    table_name=VOICE_ASSET_TABLE_ID,
    run_id=run_id,
)

print(asset_load_result)

Loaded table: voice_widget_assets
Rows: 10
GCS URI: gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/deploy/df9c8696-dcd5-40f7-ad0e-bb71eb665468/voice_widget_assets.ndjson
Seconds: 2.7557
{'table_name': 'voice_widget_assets', 'gcs_uri': 'gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/deploy/df9c8696-dcd5-40f7-ad0e-bb71eb665468/voice_widget_assets.ndjson', 'row_count': 10, 'load_seconds': 2.7557, 'job_id': '8af05f44-df77-4eaa-82ba-8ea21c4d196f'}


In [26]:
voice_test_rows = [
    {
        "test_id": str(uuid4()),
        "run_id": run_id,
        "test_name": "health_check",
        "request_method": "GET",
        "endpoint_path": "/health",
        "example_payload_json": None,
        "expected_result": "Returns service status and configuration.",
        "created_at": datetime.now(timezone.utc).isoformat(),
    },
    {
        "test_id": str(uuid4()),
        "run_id": run_id,
        "test_name": "text_chat",
        "request_method": "POST",
        "endpoint_path": "/chat",
        "example_payload_json": json.dumps(
            {
                "message": "Co to jest kolekcja w Weaviate?",
                "top_k": 5,
            },
            ensure_ascii=False,
        ),
        "expected_result": "Returns RAG answer and source chunks.",
        "created_at": datetime.now(timezone.utc).isoformat(),
    },
    {
        "test_id": str(uuid4()),
        "run_id": run_id,
        "test_name": "text_to_speech",
        "request_method": "POST",
        "endpoint_path": "/tts",
        "example_payload_json": json.dumps(
            {
                "text": "Cześć, jestem Weaviate Voice Mentor.",
                "language_code": "pl-PL",
            },
            ensure_ascii=False,
        ),
        "expected_result": "Returns base64 MP3 audio.",
        "created_at": datetime.now(timezone.utc).isoformat(),
    },
    {
        "test_id": str(uuid4()),
        "run_id": run_id,
        "test_name": "voice_chat",
        "request_method": "POST",
        "endpoint_path": "/voice-chat",
        "example_payload_json": json.dumps(
            {
                "audio_base64": "BASE64_WEBM_OPUS_AUDIO_HERE",
                "mime_type": "audio/webm;codecs=opus",
                "language_code": "pl-PL",
                "alternative_language_code": "en-US",
                "top_k": 8,
            },
            ensure_ascii=False,
        ),
        "expected_result": "Transcribes audio, runs RAG, returns answer and base64 MP3.",
        "created_at": datetime.now(timezone.utc).isoformat(),
    },
]

pd.DataFrame(voice_test_rows)

,test_id,run_id,test_name,request_method,endpoint_path,example_payload_json,expected_result,created_at
0,ddc3daad-c86d-458e-a9a0-56e1d8fe4afc,df9c8696-dcd5-40f7-ad0e-bb71eb665468,health_check,GET,/health,None,Returns service status and configuration.,2026-07-21T13:21:21.745232+00:00
1,7389180b-d628-4f02-ab10-08c181001ff3,df9c8696-dcd5-40f7-ad0e-bb71eb665468,text_chat,POST,/chat,"{""message"": ""Co to jest kolekcja w Weaviate?"",...",Returns RAG answer and source chunks.,2026-07-21T13:21:21.745312+00:00
2,9b449143-cc20-4c20-abbb-94b50707a549,df9c8696-dcd5-40f7-ad0e-bb71eb665468,text_to_speech,POST,/tts,"{""text"": ""Cześć, jestem Weaviate Voice Mentor....",Returns base64 MP3 audio.,2026-07-21T13:21:21.745337+00:00
3,52d314ff-94b7-400c-b6bb-e20a454ff9dc,df9c8696-dcd5-40f7-ad0e-bb71eb665468,voice_chat,POST,/voice-chat,"{""audio_base64"": ""BASE64_WEBM_OPUS_AUDIO_HERE""...","Transcribes audio, runs RAG, returns answer an...",2026-07-21T13:21:21.745357+00:00


In [27]:
test_load_result = batch_load_rows_to_bigquery(
    rows=voice_test_rows,
    table_ref=voice_test_table_ref,
    schema=voice_test_schema,
    table_name=VOICE_TEST_TABLE_ID,
    run_id=run_id,
)

print(test_load_result)

Loaded table: voice_widget_test_requests
Rows: 4
GCS URI: gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/deploy/df9c8696-dcd5-40f7-ad0e-bb71eb665468/voice_widget_test_requests.ndjson
Seconds: 2.3219
{'table_name': 'voice_widget_test_requests', 'gcs_uri': 'gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/deploy/df9c8696-dcd5-40f7-ad0e-bb71eb665468/voice_widget_test_requests.ndjson', 'row_count': 4, 'load_seconds': 2.3219, 'job_id': '7572089f-8719-4efb-9187-a68b45fb57a5'}


In [28]:
print(deploy_sh)

#!/usr/bin/env bash
set -euo pipefail

PROJECT_ID="leafy-guide-497515-m4"
REGION="us-central1"
SERVICE_NAME="weaviate-voice-mentor-api"
ARTIFACT_REPOSITORY="voice-mentor"

IMAGE_URI="us-central1-docker.pkg.dev/leafy-guide-497515-m4/voice-mentor/weaviate-voice-mentor-api:latest"

gcloud config set project "$PROJECT_ID"

gcloud services enable \
  run.googleapis.com \
  cloudbuild.googleapis.com \
  artifactregistry.googleapis.com \
  aiplatform.googleapis.com \
  bigquery.googleapis.com \
  speech.googleapis.com \
  texttospeech.googleapis.com

gcloud artifacts repositories describe "$ARTIFACT_REPOSITORY" \
  --location "$REGION" \
  >/dev/null 2>&1 \
  || gcloud artifacts repositories create "$ARTIFACT_REPOSITORY" \
      --repository-format=docker \
      --location "$REGION" \
      --description="Docker repository for Weaviate Voice Mentor"

gcloud builds submit \
  --tag "$IMAGE_URI" \
  .

gcloud run deploy "$SERVICE_NAME" \
  --image "$IMAGE_URI" \
  --region "$REGION" \
  --plat

In [29]:
print(lovable_embed_html[:7000])

<div id="weaviate-voice-mentor-widget"></div>

<script>
  window.WEAVIATE_MENTOR_API_URL = "PASTE_CLOUD_RUN_URL_HERE";
</script>

<script>
(function () {
  const API_URL = window.WEAVIATE_MENTOR_API_URL || "PASTE_CLOUD_RUN_URL_HERE";

  const root = document.getElementById("weaviate-voice-mentor-widget");

  if (!root) {
    console.error("Missing #weaviate-voice-mentor-widget container.");
    return;
  }

  root.innerHTML = `
    <button id="wvm-open-button">Ask Weaviate Mentor</button>

    <div id="wvm-panel" style="display:none;">
      <div id="wvm-header">
        <div>
          <strong>Weaviate Voice Mentor</strong>
          <div id="wvm-subtitle">RAG + Gemini + Google Cloud Voice</div>
        </div>
        <button id="wvm-close-button">×</button>
      </div>

      <div id="wvm-messages"></div>

      <div id="wvm-controls">
        <textarea id="wvm-input" placeholder="Zapytaj o Weaviate..."></textarea>

        <div id="wvm-buttons">
          <button id="wvm-send-butto

In [30]:
for filename, content in backend_files.items():
    print("=" * 100)
    print(filename)
    print("Characters:", len(content))

main.py
Characters: 13884
requirements.txt
Characters: 202
Dockerfile
Characters: 269
deploy.sh
Characters: 1715
smoke_test.sh
Characters: 762
README.md
Characters: 452
weaviate-voice-widget.js
Characters: 11306
widget.html
Characters: 180
lovable_embed.html
Characters: 11455


In [31]:
sql = f"""
SELECT
  asset_type,
  filename,
  gcs_uri,
  content_type,
  description,
  created_at
FROM `{voice_asset_table_ref}`
ORDER BY
  asset_type,
  filename
"""

voice_assets_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

voice_assets_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,asset_type,filename,gcs_uri,content_type,description,created_at
0,backend_code,main.py,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/x-python,FastAPI backend,2026-07-21 13:20:36.038118+00:00
1,backend_package,weaviate_voice_mentor_cloud_run_package.zip,gs://leafy-guide-497515-m4-vector-assets/weavi...,application/zip,Full backend and widget package,2026-07-21 13:20:37.717949+00:00
2,deployment_code,Dockerfile,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/plain,Cloud Run Dockerfile,2026-07-21 13:20:36.437649+00:00
3,deployment_script,deploy.sh,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/x-shellscript,Shell deployment or test script,2026-07-21 13:20:36.641923+00:00
4,deployment_script,smoke_test.sh,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/x-shellscript,Shell deployment or test script,2026-07-21 13:20:36.868142+00:00
5,documentation,README.md,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/markdown,Deployment documentation,2026-07-21 13:20:37.078520+00:00
6,other,requirements.txt,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/plain,Generated asset,2026-07-21 13:20:36.236696+00:00
7,widget_code,lovable_embed.html,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/html,Embeddable Lovable widget,2026-07-21 13:20:37.717574+00:00
8,widget_code,weaviate-voice-widget.js,gs://leafy-guide-497515-m4-vector-assets/weavi...,application/javascript,Voice widget JavaScript,2026-07-21 13:20:37.310655+00:00
9,widget_code,widget.html,gs://leafy-guide-497515-m4-vector-assets/weavi...,text/html,Embeddable Lovable widget,2026-07-21 13:20:37.499658+00:00


In [32]:
sql = f"""
SELECT
  test_name,
  request_method,
  endpoint_path,
  expected_result
FROM `{voice_test_table_ref}`
ORDER BY
  test_name
"""

voice_tests_df = bigquery_client.query(
    sql,
    location=dataset.location,
).to_dataframe()

voice_tests_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,test_name,request_method,endpoint_path,expected_result
0,health_check,GET,/health,Returns service status and configuration.
1,text_chat,POST,/chat,Returns RAG answer and source chunks.
2,text_to_speech,POST,/tts,Returns base64 MP3 audio.
3,voice_chat,POST,/voice-chat,"Transcribes audio, runs RAG, returns answer an..."


In [33]:
deployment_instructions = f"""
Manual deployment steps:

1. Download this package from GCS:

{backend_package_gcs_uri}

2. Unzip it into a local folder, for example:

weaviate_voice_mentor_cloud_run/

3. Open terminal in that folder.

4. Run:

chmod +x deploy.sh
bash deploy.sh

5. Copy the printed Cloud Run service URL.

6. Open lovable_embed.html.

7. Replace:

PASTE_CLOUD_RUN_URL_HERE

with your Cloud Run URL.

8. Paste the final lovable_embed.html content into Lovable.

9. Test endpoints:

bash smoke_test.sh YOUR_CLOUD_RUN_URL

Expected backend endpoints:

GET  /health
POST /chat
POST /tts
POST /voice-chat

Important:
The frontend does not store Google keys.
The Cloud Run service uses its Google Cloud service identity.
"""

print(deployment_instructions)


Manual deployment steps:

1. Download this package from GCS:

gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/backend/df9c8696-dcd5-40f7-ad0e-bb71eb665468/weaviate_voice_mentor_cloud_run_package.zip

2. Unzip it into a local folder, for example:

weaviate_voice_mentor_cloud_run/

3. Open terminal in that folder.

4. Run:

chmod +x deploy.sh
bash deploy.sh

5. Copy the printed Cloud Run service URL.

6. Open lovable_embed.html.

7. Replace:

PASTE_CLOUD_RUN_URL_HERE

with your Cloud Run URL.

8. Paste the final lovable_embed.html content into Lovable.

9. Test endpoints:

bash smoke_test.sh YOUR_CLOUD_RUN_URL

Expected backend endpoints:

GET  /health
POST /chat
POST /tts
POST /voice-chat

Important:
The frontend does not store Google keys.
The Cloud Run service uses its Google Cloud service identity.



In [34]:
instructions_blob_name = (
    f"{GCS_DEPLOY_PREFIX}/"
    f"{run_id}/"
    "deployment_instructions.txt"
)

deployment_instructions_gcs_uri = upload_text_to_gcs(
    deployment_instructions,
    blob_name=instructions_blob_name,
    content_type="text/plain",
)

print("Deployment instructions saved to:")
print(deployment_instructions_gcs_uri)

Deployment instructions saved to:
gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/deploy/df9c8696-dcd5-40f7-ad0e-bb71eb665468/deployment_instructions.txt


In [35]:
notebook_summary = {
    "project_id": PROJECT_ID,
    "location": LOCATION,
    "bigquery_location": dataset.location,
    "bucket_location": BUCKET_LOCATION,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "notebook": "69_w_google_cloud_weaviate_voice_widget_cloud_run.ipynb",
    "run_id": run_id,
    "service_name": SERVICE_NAME,
    "artifact_repository": ARTIFACT_REPOSITORY,
    "image_uri": image_uri,
    "source_rag": {
        "dataset": SOURCE_RAG_DATASET_ID,
        "chunk_table": SOURCE_CHUNK_TABLE_ID,
        "chunk_table_ref": source_chunk_table_ref,
        "source_chunk_count": source_summary.chunk_count,
    },
    "models": {
        "text_model": TEXT_MODEL,
        "text_embedding_model": TEXT_EMBEDDING_MODEL,
    },
    "bigquery": {
        "voice_dataset": VOICE_DATASET_ID,
        "voice_asset_table": voice_asset_table_ref,
        "voice_test_table": voice_test_table_ref,
    },
    "cloud_storage": {
        "backend_package_gcs_uri": backend_package_gcs_uri,
        "deployment_instructions_gcs_uri": deployment_instructions_gcs_uri,
        "backend_prefix": f"gs://{BUCKET_NAME}/{GCS_BACKEND_PREFIX}/{run_id}",
        "local_files_saved": False,
    },
    "generated_assets": voice_assets_df.to_dict(orient="records"),
    "test_definitions": voice_tests_df.to_dict(orient="records"),
}

summary_json = json.dumps(
    notebook_summary,
    indent=2,
    ensure_ascii=False,
    default=str,
)

summary_blob_name = (
    f"{GCS_SUMMARY_PREFIX}/"
    f"{run_id}/"
    "summary.json"
)

summary_gcs_uri = upload_text_to_gcs(
    summary_json,
    blob_name=summary_blob_name,
    content_type="application/json",
)

print("Notebook summary saved to:")
print(summary_gcs_uri)

Notebook summary saved to:
gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/summaries/df9c8696-dcd5-40f7-ad0e-bb71eb665468/summary.json


In [36]:
print("Weaviate Voice Widget Cloud Run notebook completed.")
print("=" * 100)

print("Run ID:", run_id)

print("\nSource RAG table:")
print(source_chunk_table_ref)

print("\nCloud Run service:")
print(SERVICE_NAME)

print("\nContainer image:")
print(image_uri)

print("\nGenerated package:")
print(backend_package_gcs_uri)

print("\nDeployment instructions:")
print(deployment_instructions_gcs_uri)

print("\nSummary:")
print(summary_gcs_uri)

print("\nBigQuery tables:")
print("-", voice_asset_table_ref)
print("-", voice_test_table_ref)

print("\nGenerated files:")
for row in voice_asset_rows:
    print("=" * 100)
    print(row["filename"])
    print(row["asset_type"])
    print(row["gcs_uri"])

print("\nNo local files were saved by this notebook.")

Weaviate Voice Widget Cloud Run notebook completed.
Run ID: df9c8696-dcd5-40f7-ad0e-bb71eb665468

Source RAG table:
leafy-guide-497515-m4.weaviate_mentor_rag_backend.weaviate_mentor_chunks

Cloud Run service:
weaviate-voice-mentor-api

Container image:
us-central1-docker.pkg.dev/leafy-guide-497515-m4/voice-mentor/weaviate-voice-mentor-api:latest

Generated package:
gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/backend/df9c8696-dcd5-40f7-ad0e-bb71eb665468/weaviate_voice_mentor_cloud_run_package.zip

Deployment instructions:
gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/deploy/df9c8696-dcd5-40f7-ad0e-bb71eb665468/deployment_instructions.txt

Summary:
gs://leafy-guide-497515-m4-vector-assets/weaviate-voice-widget-cloud-run/summaries/df9c8696-dcd5-40f7-ad0e-bb71eb665468/summary.json

BigQuery tables:
- leafy-guide-497515-m4.weaviate_voice_widget_cloud_run.voice_widget_assets
- leafy-guide-497515-m4.weaviate_voice_widget_cloud_run.voice_